# BP4 Gate 6 — Productization, Monitoring & Governance
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Purpose
Implements Master Execution Plan Section 8 Gate 6: "Productization, Monitoring & Governance." Exit
criteria: "MODEL_CARD.md, CHANGELOG.md, pytest suite, CI entry, Evidence Ledger row — governance
artifacts exist BEFORE the BP is marked complete." Mirrors BP1/BP2/BP3 Gate 6's structure and rigor,
adapted for BP4's real shape: a deterministic aggregation/reporting pipeline with no supervised
target and no trained model, rather than a classifier.

## Why this gate looks different from BP1/BP2/BP3's
BP4 trains nothing. Its "champion" (Gate 3) is the fastest **correct** real execution engine among
five candidate implementations of the same deterministic aggregation, not a model selected by a
predictive metric. Three consequences follow, each a deliberate adaptation rather than an omission:
- **No `model_inventory_entry.json` equivalent.** There is no model to inventory. This gate folds
  all of Gate 6's own accumulation into `gate6_governance_summary.json` alone, and states the SR
  11-7 model-inventory compliance touchpoint as `NOT_APPLICABLE` plainly, in both
  `gate6_governance_summary.json` and MODEL_CARD.md, rather than fabricating an inventory record for
  a model that does not exist.
- **`MODEL_CARD.md` keeps its filename** for cross-BP naming consistency with the Master Plan's own
  literal Gate 6 output list, but its opening note states explicitly that it documents a real
  deterministic pipeline (execution-engine benchmark, statistical validation, reporting layer), never
  a trained model's architecture or predictive performance — nothing in this document is invented
  ML-model framing.
- **No shared-module extension was needed.** Unlike BP3 (whose Gate 6 extracted duplicated
  CANDIDATES/preprocessing logic out of Gates 3/4/5 into `bp3_escalation_features.py` for the first
  time), BP4's `src/features/bp4_journey_features.py` already centralized `CLUSTER_KEY`,
  `BARRED_JOURNEY_COLUMNS`, and `NULL_SENTINEL_MAP` at Gate 2 (HYPER). Gate 3's five execution-engine
  functions (`run_pandas_groupby`, `run_polars_eager`, `run_polars_lazy`, `run_polars_lazy_streaming`,
  `run_duckdb_sql`) are one-off benchmark candidates, never reused or triplicated by Gates 4/5, so
  there was nothing to de-duplicate. This gate still delivers BP4's **first-ever test coverage**
  (`pytest tests/` previously only exercised BP1/BP2/BP3's own tests) — two new test files, described
  below.

## What this gate does, concretely
This notebook computes nothing new about the journey data itself — every number it reports was
already computed and recorded by BP4 Gates 1-5's own real runs. Gate 6 performs four real actions,
all executed on this machine when you run it:
1. **Reads Gates 1-5's real artifacts live** (the shared config YAML plus `policy.json`,
   `gate2_feature_lineage.csv`, `gate3_benchmark_results.csv`, `gate5_decision_layer_summary.json`,
   `gate5_cluster_decision_report.csv`) and cross-checks their internal consistency: the champion
   pipeline name recorded in the config's top-level `champion_pipeline` (Gate 3) and
   `performance_report.champion_pipeline` (Gate 4) must agree; the real cluster count recorded by
   Gate 1 (`policy.json`), Gate 2 (`n_clusters` + `cluster_count_matches_gate1`), and Gate 5
   (`n_clusters_reported`) must all describe the identical population; and Gate 5's reused
   `gate4_population_mean_lag_reference` must match Gate 4's own recorded
   `mean_response_lag_days.point_estimate` to within floating-point tolerance (both are read from the
   identical live source, so any drift here would be a real bug).
2. **Detects live, from `gate3_benchmark_results.csv`, two kinds of open item** — never hardcoded by
   candidate name, so this still works correctly on a future re-run against the full 1,048,575-row
   real dataset, where timings (and potentially the champion itself) can differ:
   - any candidate whose `status` is not `"CORRECT"` (BP4's real Gate 3 run had none — all 5
     candidates passed correctness — but the check exists unconditionally for a future re-run);
   - a **BP4-specific category**: a `CORRECT` candidate whose `min_seconds` exceeds 2x the champion's
     own real `min_seconds`. BP4's real Gate 3 run flags two — `duckdb_sql` and `pandas_groupby` —
     both real, expected variation across execution-engine designs at this row count, not a defect;
     reported here as an honest open item for future re-benchmarking rather than left unrecorded.
3. **Runs the project's full pytest suite for real**, via `subprocess` (`pytest tests/ -v
   --tb=short`, the exact invocation `.github/workflows/ci.yml` uses), and the static
   notebook-syntax audit (`scripts/check_notebook_syntax.py`) for real, also via `subprocess`. Both
   are real governance integrity gates — their pass/fail result is not assumed or simulated, and
   both are asserted as structural checks at the end of this notebook. This is the suite's **first
   run with real BP4 coverage**: two new test files delivered alongside this notebook
   (`tests/bp4_customer_journey_analytics/test_bp4_journey_features.py`,
   `tests/bp4_customer_journey_analytics/test_gate_artifacts.py`) give
   `src/features/bp4_journey_features.py` and BP4's cross-gate artifacts their first-ever coverage.
4. **Deterministically generates `MODEL_CARD.md` and `CHANGELOG.md`** from the real values loaded in
   step 1 (an f-string template — no GenAI-authored freeform text, per the project's zero-fabrication
   rule) and writes a `gate6_governance` block to the shared config via the same order-independent
   `bp1_config_sync.write_gate_block()` helper Gates 2-5 already use. This gate also sets the config's
   `status` field to `"gate6_complete"` directly — see "The `status` field fix" below.

## The `status` field fix
BP4's config `status` field was set to `"gate1_confirmed"` by Gate 1 and never updated by Gates 2-5
(unlike BP3's own convention of each gate appending its own `_gateN_confirmed` suffix). This is a
real, pre-existing inconsistency in BP4's own config, not something this gate's design introduces.
Rather than retroactively editing Gates 2-5's already-closed, already real-run-confirmed notebooks —
which the project's standing discipline avoids — Gate 6 fixes it going forward: it is the gate
literally responsible for marking the BP complete, so it sets `status: "gate6_complete"` directly,
matching the field's own pre-existing documented comment/enum
(`# not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete`). Gates 2-5's own real, already-verified
work is left untouched.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: every number in MODEL_CARD.md / CHANGELOG.md is read
  live from Gates 1-5's own already-recorded real artifacts, or computed live from them (e.g. the
  Known Limitations items are *detected* live from `gate3_benchmark_results.csv`, not hardcoded from
  a prior conversation) — never typed in as a remembered figure.
- **HYPER**: no BP4 shared module needed extension at this gate (see "Why this gate looks different"
  above) — `src/features/bp4_journey_features.py` is delivered unchanged; only its first real test
  coverage is new.
- **Idempotent**: re-running overwrites this gate's artifacts and MODEL_CARD.md/CHANGELOG.md, and
  appends/replaces only the `gate6_governance` block plus the `status` field in
  `configs/bp4_customer_journey_analytics.yaml`, without touching Gates 1-5's own blocks.

## Outputs (idempotent overwrite-in-place)
- `reports/bp4_customer_journey_analytics/MODEL_CARD.md`
- `reports/bp4_customer_journey_analytics/CHANGELOG.md`
- `notebooks/bp4_customer_journey_analytics/artifacts/gate6_governance_summary.json`
- `notebooks/bp4_customer_journey_analytics/artifacts/gate6_pytest_output.log` (full captured
  stdout+stderr of the real pytest run, for audit trail)
- `notebooks/bp4_customer_journey_analytics/artifacts/gate6_notebook_syntax_check_output.log`
- `configs/bp4_customer_journey_analytics.yaml` — `gate6_governance` block appended/updated; `status`
  set to `"gate6_complete"`

## Prerequisites
BP4 Gates 1-5 must all have been real-run at least once — this notebook reads and cross-checks all
five gates' recorded artifacts and raises a clear `AssertionError` naming whichever one is missing.
Two new test files must already be in place under `tests/bp4_customer_journey_analytics/` (delivered
alongside this notebook, not written by it):
`test_bp4_journey_features.py` and `test_gate_artifacts.py`.

## If a structural check below fails
It raises `AssertionError` naming the failing check — including if the real pytest suite has any
failures/errors, or if the real static notebook-syntax audit fails on any notebook. Do not silence
it. Note that MODEL_CARD.md and CHANGELOG.md are still written even in that case (so the real failure
is documented in the Governance & Testing section rather than hidden), but Gate 6 is not considered
complete until the final `[ALL CHECKS PASSED]` line prints.

## Known limitations carried forward (read live below, not from memory)
- **`pandas_groupby` and `duckdb_sql` are more than 2x slower than the champion `polars_eager`** on
  this real run, despite both passing correctness (`status: CORRECT`). Not a defect — expected
  variation across execution-engine designs; champion selection is unaffected by construction
  (`min_seconds` ranking among `CORRECT` candidates only). Worth re-checking against the full
  1,048,575-row real dataset, where the ranking can shift.
- **Gate 5's `review_priority_score/tier` is a BP4-local reporting flag, not BP7's cross-BP decision
  engine** (documented in Gate 5's own `bp7_decision_engine_boundary` compliance note, carried
  forward into this gate's MODEL_CARD.md verbatim) — it may become one real input BP7 later combines,
  but is never presented as BP7's own decision.
- **The Gold-layer `common_taxonomy_bucket` BANKING77 overlay real-covers only 6.55% of rows**
  (Gate 1 finding, carried forward unchanged) — used only as an optional secondary overlay in Gate
  5's tier reporting, never as BP4's primary journey-grouping dimension or presented as
  population-level coverage.
- **BP4's config `status` field was never updated past `"gate1_confirmed"` by Gates 2-5** — a real,
  pre-existing inconsistency this gate fixes going forward (see "The `status` field fix" above)
  without retroactively touching any already-closed gate's own notebook.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP4 Gate 6 (Productization, Monitoring & Governance)
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the "
        "project tree (expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp4_customer_journey_analytics"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import subprocess  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import pandas as pd  # noqa: E402
import yaml  # noqa: E402

from features.bp4_journey_features import BARRED_JOURNEY_COLUMNS, CLUSTER_KEY  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

# ============================================================
# SECTION 4: Load Gates 1-5's real artifacts LIVE (never hardcoded) + cross-gate consistency checks.
# Unlike BP3, BP4 registers no supervised model: there is no model_inventory_entry.json equivalent
# to load or extend here (see markdown - this Gate 6 folds all of that accumulation into
# gate6_governance_summary.json alone, with an honest NOT_APPLICABLE for the SR 11-7 model-inventory
# compliance touchpoint below).
# ============================================================
bp4_config_path = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"
assert bp4_config_path.exists(), f"[CHECK FAILED] {bp4_config_path} not found - run BP4 Gate 1 first."
with open(bp4_config_path, "r", encoding="utf-8") as f:
    bp4_config = yaml.safe_load(f)

for _required_key, _gate_label in (
    ("journey_definition", "Gate 1"),
    ("journey_row_count", "Gate 2"),
    ("champion_pipeline", "Gate 3"),
    ("mean_response_lag_days", "Gate 4"),
    ("n_clusters_reported", "Gate 5"),
):
    assert bp4_config.get(_required_key) is not None, (
        f"[CHECK FAILED] '{_required_key}' is missing from {bp4_config_path.name} - run BP4 "
        f"{_gate_label} first."
    )

policy_path = ARTIFACTS_DIR / "policy.json"
assert policy_path.exists(), f"[CHECK FAILED] {policy_path} not found - run BP4 Gate 1 first."
with open(policy_path, "r", encoding="utf-8") as f:
    policy = json.load(f)

gate2_lineage_path = ARTIFACTS_DIR / "gate2_feature_lineage.csv"
assert gate2_lineage_path.exists(), f"[CHECK FAILED] {gate2_lineage_path} not found - run BP4 Gate 2 first."
gate2_lineage_df = pd.read_csv(gate2_lineage_path)

gate3_csv_path = ARTIFACTS_DIR / "gate3_benchmark_results.csv"
assert gate3_csv_path.exists(), f"[CHECK FAILED] {gate3_csv_path} not found - run BP4 Gate 3 first."
gate3_df = pd.read_csv(gate3_csv_path)

gate5_summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_summary_path.exists(), f"[CHECK FAILED] {gate5_summary_path} not found - run BP4 Gate 5 first."
with open(gate5_summary_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)

gate5_csv_path = ARTIFACTS_DIR / "gate5_cluster_decision_report.csv"
assert gate5_csv_path.exists(), f"[CHECK FAILED] {gate5_csv_path} not found - run BP4 Gate 5 first."
gate5_df = pd.read_csv(gate5_csv_path, low_memory=False)

# BP4 Gate 2 (like BP1/BP2/BP3's own Gate 2) writes no own JSON summary/timestamp - its real
# completion time is the issue-cluster-summary Gold Parquet's own real filesystem modification
# time (a live-read fact, not a remembered/typed one), same convention every prior BP's Gate 6 used.
cluster_summary_gold_path = PROJECT_ROOT / bp4_config["cluster_summary_gold_path"]
gate2_mtime_utc = (
    datetime.fromtimestamp(cluster_summary_gold_path.stat().st_mtime, tz=timezone.utc).isoformat()
    if cluster_summary_gold_path.exists()
    else None
)

# Cross-gate champion (of the aggregation-pipeline benchmark) consistency - must agree everywhere
# it is recorded.
CHAMPION_PIPELINE = bp4_config["champion_pipeline"]
_champion_sources = {
    "config champion_pipeline (Gate 3)": bp4_config["champion_pipeline"],
    "config performance_report.champion_pipeline (Gate 4)": bp4_config["performance_report"][
        "champion_pipeline"
    ],
}
_champion_mismatches = {k: v for k, v in _champion_sources.items() if v != CHAMPION_PIPELINE}
assert not _champion_mismatches, (
    f"[CHECK FAILED] Champion pipeline disagrees across recorded artifacts: {_champion_mismatches} "
    f"(expected '{CHAMPION_PIPELINE}' everywhere)."
)

# Cross-gate cluster-count consistency - Gate 1's own live-verified n_clusters (recorded in
# policy.json's issue_cluster_stats), Gate 2's own n_clusters + cluster_count_matches_gate1 flag,
# and Gate 5's own n_clusters_reported must all describe the identical real cluster population.
_gate1_n_clusters = policy["live_checks"]["issue_cluster_stats"]["n_clusters"]
_gate2_n_clusters = bp4_config["n_clusters"]
_gate5_n_clusters = bp4_config["n_clusters_reported"]
assert bp4_config["cluster_count_matches_gate1"] is True, (
    "[CHECK FAILED] Gate 2's own cluster_count_matches_gate1 flag is not True - Gate 2 already "
    "detected a mismatch against Gate 1 on its own run; Gate 6 will not paper over that."
)
assert _gate1_n_clusters == _gate2_n_clusters == _gate5_n_clusters, (
    f"[CHECK FAILED] Cluster counts disagree across gates: Gate 1 (policy.json) = {_gate1_n_clusters}, "
    f"Gate 2 (config n_clusters) = {_gate2_n_clusters}, Gate 5 (config n_clusters_reported) = "
    f"{_gate5_n_clusters} - these must describe the identical real cluster population."
)

# Gate 4's own recorded mean-response-lag point estimate and Gate 5's reused reference to it must
# be read from the identical live source (Gate 5 re-derives this live at run time - never retypes
# it - so any drift here is a real bug, not floating-point noise).
_gate4_mean_lag = bp4_config["mean_response_lag_days"]["point_estimate"]
_gate5_mean_lag_reference = bp4_config["gate4_population_mean_lag_reference"]
_mean_lag_diff = abs(_gate4_mean_lag - _gate5_mean_lag_reference)
assert _mean_lag_diff < 1e-9, (
    f"[CHECK FAILED] Gate 4's recorded mean_response_lag_days.point_estimate ({_gate4_mean_lag}) "
    f"does not match Gate 5's reused gate4_population_mean_lag_reference ({_gate5_mean_lag_reference}) "
    f"(diff={_mean_lag_diff})."
)

print(
    f"[OK] Champion pipeline '{CHAMPION_PIPELINE}' confirmed consistent across "
    f"{len(_champion_sources)} independently recorded real artifacts. Cluster count ({_gate2_n_clusters}) "
    "and Gate 4->Gate 5 mean-lag reference confirmed consistent across gates."
)

# ============================================================
# SECTION 5: Detect open Gate 3 items LIVE from the real benchmark results - TWO categories, never
# hardcoded by candidate name (this must still work correctly on a future re-run against the full
# 1,048,575-row real dataset, where timings and even which candidate wins can differ):
#   (a) any candidate whose status is NOT "CORRECT"
#   (b) a CORRECT candidate whose min_seconds exceeds a live-computed multiplier of the champion's
#       own min_seconds - a BP4-specific addition (BP4 has no predictive-metric anomaly category the
#       way BP3's near-random-PR-AUC/near-zero-recall checks do, since Gate 3 here benchmarks
#       deterministic execution engines, not classifiers - see markdown).
# ============================================================
SLOW_CANDIDATE_MULTIPLIER = 2.0

correct_mask = gate3_df["status"] == "CORRECT"
failed_candidate_rows = gate3_df[~correct_mask]
correct_rows = gate3_df[correct_mask]

champion_row = correct_rows[correct_rows["candidate"] == CHAMPION_PIPELINE]
assert len(champion_row) == 1, (
    f"[CHECK FAILED] Expected exactly one CORRECT row for champion '{CHAMPION_PIPELINE}' in "
    f"{gate3_csv_path.name}, found {len(champion_row)}."
)
champion_min_seconds = float(champion_row.iloc[0]["min_seconds"])
slow_threshold_seconds = round(SLOW_CANDIDATE_MULTIPLIER * champion_min_seconds, 6)
slow_candidate_rows = correct_rows[
    (correct_rows["candidate"] != CHAMPION_PIPELINE) & (correct_rows["min_seconds"] > slow_threshold_seconds)
]

known_limitation_lines = []
for _, row in failed_candidate_rows.iterrows():
    known_limitation_lines.append(
        f"- **`{row['candidate']}` (FAILED Gate 3's correctness benchmark)**: real recorded status - "
        f"`{row['status']}`. Champion selection excludes any non-`CORRECT` candidate by construction "
        '(`gate3_df[gate3_df["status"] == "CORRECT"]`), so this failure never had a path to silently '
        "becoming the champion."
    )
if failed_candidate_rows.empty:
    known_limitation_lines.append(
        f"- No candidate failed Gate 3's correctness benchmark on this real run (all "
        f"{len(gate3_df)} CORRECT)."
    )
for _, row in slow_candidate_rows.iterrows():
    known_limitation_lines.append(
        f"- **`{row['candidate']}`**: real min_seconds {float(row['min_seconds']):.6f} is more than "
        f"{SLOW_CANDIDATE_MULTIPLIER}x the champion `{CHAMPION_PIPELINE}`'s real min_seconds "
        f"({champion_min_seconds:.6f}, threshold {slow_threshold_seconds:.6f}) despite passing "
        "correctness - not a defect, expected variation across execution-engine designs at this row "
        "count; noted here for future re-benchmarking against the full real dataset, where the ranking "
        "can shift."
    )
if slow_candidate_rows.empty:
    known_limitation_lines.append(
        f"- No CORRECT candidate exceeded {SLOW_CANDIDATE_MULTIPLIER}x the champion's real min_seconds "
        f"on this run (threshold: {slow_threshold_seconds:.6f}s)."
    )

print(
    f"[OK] Gate 3 open-item detection (live): {len(failed_candidate_rows)} failed candidate row(s), "
    f"{len(slow_candidate_rows)} slow-but-correct candidate row(s) (>{SLOW_CANDIDATE_MULTIPLIER}x champion)."
)

# ============================================================
# SECTION 6: Run the project's full pytest suite for REAL, via subprocess (exact CI invocation).
# This is the suite's first run with real BP4 coverage (src/features/bp4_journey_features.py's
# existing Gate 2-built functions, exercised for the first time by 2 new test files delivered
# alongside this notebook - see markdown).
# ============================================================
print("\n[GATE6] Running the real pytest suite (pytest tests/ -v --tb=short)...")
pytest_cmd = [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"]
pytest_result = subprocess.run(
    pytest_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
pytest_log_path = ARTIFACTS_DIR / "gate6_pytest_output.log"
with open(pytest_log_path, "w", encoding="utf-8") as f:
    f.write(pytest_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(pytest_result.stderr)
print(f"[SAVED] {pytest_log_path.relative_to(PROJECT_ROOT)} (pytest exit code {pytest_result.returncode})")

pytest_summary_line = ""
for _line in reversed(pytest_result.stdout.splitlines()):
    if "==" in _line and any(_k in _line for _k in ("passed", "failed", "error", "no tests ran")):
        pytest_summary_line = _line.strip(" =")
        break
pytest_counts = {"passed": 0, "failed": 0, "skipped": 0, "errors": 0, "xfailed": 0, "xpassed": 0}
for _count_str, _label in re.findall(
    r"(\d+)\s+(passed|failed|skipped|error|errors|xfailed|xpassed)", pytest_summary_line
):
    _key = "errors" if _label == "error" else _label
    pytest_counts[_key] = int(_count_str)
pytest_total = sum(pytest_counts.values())
pytest_all_passed = (
    pytest_result.returncode == 0
    and pytest_counts["failed"] == 0
    and pytest_counts["errors"] == 0
    and (pytest_counts["passed"] + pytest_counts["xpassed"]) > 0
)
print(
    f"[RESULT] pytest: {pytest_summary_line!r} -> parsed counts {pytest_counts} "
    f"(all_passed={pytest_all_passed})"
)

# ============================================================
# SECTION 7: Run the static notebook-syntax audit for REAL, via subprocess (nbformat + ast +
# pyflakes - never executes any notebook's code, per the project's execution-boundary rule). This
# is project-wide (scans everything under notebooks/), so it covers every BP's notebooks, not just
# BP4's.
# ============================================================
print("\n[GATE6] Running the real static notebook-syntax audit (scripts/check_notebook_syntax.py)...")
syntax_check_cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "check_notebook_syntax.py")]
syntax_check_result = subprocess.run(
    syntax_check_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
syntax_log_path = ARTIFACTS_DIR / "gate6_notebook_syntax_check_output.log"
with open(syntax_log_path, "w", encoding="utf-8") as f:
    f.write(syntax_check_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(syntax_check_result.stderr)
print(f"[SAVED] {syntax_log_path.relative_to(PROJECT_ROOT)} (exit code {syntax_check_result.returncode})")

syntax_pass_lines = [line for line in syntax_check_result.stdout.splitlines() if line.startswith("[PASS]")]
syntax_fail_lines = [line for line in syntax_check_result.stdout.splitlines() if line.startswith("[FAIL]")]
notebook_syntax_all_passed = (
    syntax_check_result.returncode == 0 and len(syntax_fail_lines) == 0 and len(syntax_pass_lines) > 0
)
print(
    f"[RESULT] Notebook syntax check: {len(syntax_pass_lines)} passed, {len(syntax_fail_lines)} failed "
    f"(all_passed={notebook_syntax_all_passed})"
)

# ============================================================
# SECTION 8: Generate MODEL_CARD.md - deterministic f-string template from ONLY the real values
# loaded/computed above (zero-fabrication rule: no GenAI-authored freeform text). Filename kept for
# cross-BP naming consistency with the Master Plan's own literal Gate 6 output list, but the
# opening note below states plainly that this documents a real deterministic aggregation/reporting
# pipeline, not a trained ML model (BP4 has no supervised target and trains nothing).
# ============================================================
_now_utc = datetime.now(timezone.utc).isoformat()
_tier_rollup = gate5_summary["tier_rollup"]
_tier_rollup_lines = "\n".join(
    f"  | {t['review_priority_tier']} | {t['n_clusters']:,} | {t['n_complaint_rows_covered']:,} | "
    f"{t['mean_banking77_coverage_fraction']} |"
    for t in _tier_rollup
)
_candidates_all = gate3_df.assign(_is_correct=(gate3_df["status"] == "CORRECT")).sort_values(
    ["_is_correct", "min_seconds"], ascending=[False, True], kind="mergesort", na_position="last"
)
_candidate_table_lines = "\n".join(
    f"  | {r.candidate} | {r.status} | {r.min_seconds if pd.notna(r.min_seconds) else 'n/a'} | "
    f"{r.mean_seconds if pd.notna(r.mean_seconds) else 'n/a'} | {'YES' if r.is_champion else ''} |"
    for r in _candidates_all.itertuples()
)
_lineage_table_lines = "\n".join(
    f"  | {r.engineered_feature} | {r.source_column} |" for r in gate2_lineage_df.itertuples()
)
_perf = bp4_config["performance_report"]

MODEL_CARD_MD = f"""# Model Card — BP4 Customer Journey Analytics

*Generated {_now_utc} by `bp4_customer_journey_analytics_g6_productization_monitoring_governance.ipynb`,
deterministically, from real values recorded by BP4 Gates 1-5's own real runs on this machine. No field
below was authored freeform or by a generative model (project zero-fabrication rule).*

**A note on this document's name:** BP4 has no supervised target and trains no predictive model - it
is a deterministic aggregation and reporting pipeline over real, already-verified issue-cluster
statistics. "MODEL_CARD.md" is kept as the filename only for naming consistency with the Master
Plan's own literal Gate 6 output list across every BP; every section below documents the real
pipeline (its execution-engine benchmark, its statistical validation, its reporting layer), never a
trained model's architecture, training run, or predictive performance, none of which exists here.

## Pipeline Details
- **Champion aggregation pipeline (Gate 3, real benchmark):** `{CHAMPION_PIPELINE}` — real min_seconds
  {champion_min_seconds:.6f}s, {SLOW_CANDIDATE_MULTIPLIER}x-champion slow-candidate threshold
  {slow_threshold_seconds:.6f}s
- **Candidates evaluated (real Gate 3 correctness + timing benchmark, all {len(gate3_df)}):**

  | candidate | status | min_seconds | mean_seconds | champion |
  |---|---|---|---|---|
{_candidate_table_lines}

- **Real speedup over the slowest correctness-passing baseline:** {_perf['speedup_factor']:.4f}x
  ({_perf['baseline_seconds']}s -> {_perf['champion_seconds']}s)
- **Journey-grouping key (real, CFPB's own schema, never a fabricated identifier):**
  `{", ".join(CLUSTER_KEY)}`
- **Barred from every BP4 journey-grouping key (Gate 1 scope decision):**
  `{", ".join(BARRED_JOURNEY_COLUMNS)}`

## Intended Use
- **Journey definition:** {bp4_config['journey_definition']['naming_commitment']}
- **Unit 1 (complaint-event journey):** {bp4_config['journey_definition']['unit_1_complaint_event_journey']}
- **Unit 2 (issue-cluster journey):** {bp4_config['journey_definition']['unit_2_issue_cluster_journey']}
- **Out of scope:** no per-customer or per-consumer journey, cohort, retention, or survival-style
  analysis is buildable or intended — this real extract carries no customer/consumer identifier
  (Gate 1, live-verified).

## Training Data
*Not applicable — BP4 trains no model. The real data this pipeline aggregates:*
- **Source:** real CFPB extract, `{bp4_config['journey_event_gold_path']}`
- **Real journey-event row count (Gate 2):** {bp4_config['journey_row_count']:,} (matches raw:
  {bp4_config['journey_row_count_matches_raw']})
- **Real issue-cluster count (Gate 2, matches Gate 1: {bp4_config['cluster_count_matches_gate1']}):**
  {bp4_config['n_clusters']:,} clusters, {bp4_config['n_recurring_clusters']:,} recurring
- **Gate 2 (real run, Gold layer's own file modification time
  {gate2_mtime_utc if gate2_mtime_utc else 'not found on this run'}):** issue-cluster-summary Gold
  layer written from this real journey-event population.
- **Feature/aggregation lineage (Gate 2, real):**

  | engineered feature | source column(s) |
  |---|---|
{_lineage_table_lines}

## Evaluation Data & Results
- **Statistical validation (Gate 4, real, {bp4_config['n_bootstrap']:,}-resample bootstrap, 95% CI):**
  - Mean response lag (days): point estimate {bp4_config['mean_response_lag_days']['point_estimate']},
    95% CI [{bp4_config['mean_response_lag_days']['ci_95_low']},
    {bp4_config['mean_response_lag_days']['ci_95_high']}]
  - Recurring-cluster rate: point estimate {bp4_config['recurring_cluster_rate']['point_estimate']},
    95% CI [{bp4_config['recurring_cluster_rate']['ci_95_low']},
    {bp4_config['recurring_cluster_rate']['ci_95_high']}]
  - Mean cluster size: point estimate {bp4_config['mean_cluster_size']['point_estimate']}, 95% CI
    [{bp4_config['mean_cluster_size']['ci_95_low']}, {bp4_config['mean_cluster_size']['ci_95_high']}]
  - Reproducibility confirmed (Gate 4, real re-run comparison): {bp4_config['reproducibility_confirmed']}
- **Decision/reporting layer (Gate 5, real, {gate5_summary['n_clusters']:,} clusters scored):**
  {gate5_summary['n_with_reason_codes']:,} clusters carry at least one triggered review flag;
  {gate5_summary['grounding_failures']} reason-code grounding failures recorded.
- **Real tier rollup (Gate 5):**

  | tier | n_clusters | n_complaint_rows_covered | mean BANKING77 coverage |
  |---|---|---|---|
{_tier_rollup_lines}

- **Row-coverage cross-check:** Gate 5's real tier rollup sums to
  {gate5_summary['total_rows_covered_check']:,} rows, matching Gate 1/2's own real
  journey_row_count ({bp4_config['journey_row_count']:,}):
  {gate5_summary['total_rows_covered_matches_gate1']}

## Explainability
- **Method:** not applicable in the SHAP/feature-attribution sense (no predictive model). Every Gate
  5 review flag instead carries a grounded, deterministic `reason_codes`/`reason_evidence` pair — the
  real number that triggered each flag (recurring_flag from Gate 2's own is_recurring_cluster;
  elevated_lag_flag from a live comparison against Gate 4's own bootstrap point estimate;
  high_volume_flag from a live-computed 90th percentile of the real cluster population) — never a
  black-box score.

## Ethical Considerations / Compliance Touchpoints
- **CFPB supervisory & complaint-handling standards (Gate 1):** {policy['compliance_touchpoint']['statement']}
- **UDAAP language review (Gate 5):** {gate5_summary['compliance_touchpoint']['udaap_language_review']}
- **NIST AI RMF Measure/Manage (Gate 5):**
  {gate5_summary['compliance_touchpoint']['nist_ai_rmf_measure_manage']}
- **BP7 decision-engine boundary (Gate 5):**
  {gate5_summary['compliance_touchpoint']['bp7_decision_engine_boundary']}
- **ECOA/Reg B disparate-impact applicability (Gate 4):**
  {bp4_config['ecoa_disparate_impact_applicability']} — ECOA/Reg B is not a BP4 compliance touchpoint
  per Master Plan Section 9 (mapped to BP1, BP2, BP3, BP7 only); `Tags` remains barred from every
  BP4 journey-grouping key anyway, as a conservative scope decision (Gate 1).
- **Model inventory (SR 11-7):** NOT_APPLICABLE — BP4 registers no trained model, only this
  benchmarked, statistically-validated execution/reporting pipeline; there is no
  `model_inventory_entry.json` equivalent for BP4 (see gate6_governance_summary.json instead, which
  folds this Gate's own accumulation of Gates 1-5's real recorded facts).
- **GenAI API used in BP4:** {gate5_summary['compliance_touchpoint']['genai_api_used']} (scope decision
  confirmed by user {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## Known Limitations
{chr(10).join(known_limitation_lines)}

## Governance & Testing (this Gate 6 run, {_now_utc})
- **pytest suite** (`pytest tests/ -v --tb=short`): {pytest_summary_line!r} → parsed as
  {pytest_counts} (exit code {pytest_result.returncode}, all_passed={pytest_all_passed}). First run
  with real BP4 coverage — `src/features/bp4_journey_features.py` needed no Gate 6 extension (its
  CLUSTER_KEY/BARRED_JOURNEY_COLUMNS/NULL_SENTINEL_MAP were already centralized at Gate 2); 2 new
  test files delivered alongside this notebook
  (`tests/bp4_customer_journey_analytics/test_bp4_journey_features.py`,
  `tests/bp4_customer_journey_analytics/test_gate_artifacts.py`) give it its first-ever coverage.
- **Static notebook audit** (`scripts/check_notebook_syntax.py` — nbformat + ast + pyflakes, static
  only, nothing executed): {len(syntax_pass_lines)} passed / {len(syntax_fail_lines)} failed (exit
  code {syntax_check_result.returncode}, all_passed={notebook_syntax_all_passed})
- Full logs:
  `notebooks/bp4_customer_journey_analytics/artifacts/gate6_pytest_output.log`,
  `gate6_notebook_syntax_check_output.log`

## Change History
See `CHANGELOG.md` in this same folder.
"""

model_card_path = REPORTS_DIR / "MODEL_CARD.md"
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(MODEL_CARD_MD)
print(f"[SAVED] {model_card_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Generate CHANGELOG.md - one real, dated entry per gate (timestamps read live from
# each gate's own recorded artifact; Gate 2's own Gold-layer file mtime where no JSON timestamp
# exists).
# ============================================================
CHANGELOG_MD = f"""# CHANGELOG — BP4 Customer Journey Analytics

All dates below are real UTC timestamps read live from each gate's own recorded artifact at the
moment this Gate 6 notebook was run ({_now_utc}) — not typed in from memory.

## [Gate 6] Productization, Monitoring & Governance — {_now_utc}
- pytest suite: {pytest_counts['passed']} passed, {pytest_counts['failed']} failed,
  {pytest_counts['skipped']} skipped, {pytest_counts['errors']} errors ({pytest_total} total) - first
  run with real BP4 coverage (2 new test files; no shared-module extension was needed)
- Static notebook-syntax audit: {len(syntax_pass_lines)}/\
{len(syntax_pass_lines) + len(syntax_fail_lines)} notebooks passed
- MODEL_CARD.md and this CHANGELOG.md generated deterministically from Gates 1-5's real recorded \
artifacts (MODEL_CARD.md reframed for a deterministic pipeline, not a trained model - BP4 has none)
- `gate6_governance` block written to `configs/bp4_customer_journey_analytics.yaml`; config `status`
  set to `"gate6_complete"`
- Open items detected live this run: {len(failed_candidate_rows)} failed candidates,
  {len(slow_candidate_rows)} slow-but-correct candidates (>{SLOW_CANDIDATE_MULTIPLIER}x champion)

## [Gate 5] Decision Layer & Reporting — {gate5_summary['generated_at_utc']}
- {gate5_summary['n_clusters']:,} real clusters scored; {gate5_summary['n_with_reason_codes']:,} with
  at least one triggered review flag ({gate5_summary['grounding_failures']} grounding failures)
- Real tier rollup: {", ".join(f"{t['review_priority_tier']}={t['n_clusters']:,}" for t in _tier_rollup)}
- Row-coverage sum matches Gate 1's real journey_row_count:
  {gate5_summary['total_rows_covered_matches_gate1']}
- No GenAI API call (scope decision confirmed by user
  {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## [Gate 4] Statistical Validation — real bootstrap CIs recorded in
`configs/bp4_customer_journey_analytics.yaml`
- Mean response lag (days): {bp4_config['mean_response_lag_days']['point_estimate']} 95% CI
  [{bp4_config['mean_response_lag_days']['ci_95_low']}, {bp4_config['mean_response_lag_days']['ci_95_high']}]
- Recurring-cluster rate: {bp4_config['recurring_cluster_rate']['point_estimate']} 95% CI
  [{bp4_config['recurring_cluster_rate']['ci_95_low']}, {bp4_config['recurring_cluster_rate']['ci_95_high']}]
- Reproducibility confirmed: {bp4_config['reproducibility_confirmed']} | ECOA/Reg B applicability:
  {bp4_config['ecoa_disparate_impact_applicability']}

## [Gate 3] Aggregation-Pipeline Benchmark & Champion Selection
- Champion: `{CHAMPION_PIPELINE}` (real min_seconds {champion_min_seconds:.6f}s, real speedup
  {_perf['speedup_factor']:.4f}x over the slowest correctness-passing baseline)
- Candidates evaluated: {len(gate3_df)}; failed: {len(failed_candidate_rows)}
- Open items detected live this run from `gate3_benchmark_results.csv`: {len(failed_candidate_rows)}
  failed, {len(slow_candidate_rows)} slow-but-correct (see MODEL_CARD.md Known Limitations)

## [Gate 2] Data Verification & Feature/Taxonomy Engineering — \
{gate2_mtime_utc if gate2_mtime_utc else 'not found on this run'}
(file modification time of the issue-cluster-summary Gold layer; Gate 2 does not record its own JSON
timestamp)
- Real journey_row_count: {bp4_config['journey_row_count']:,} (matches raw:
  {bp4_config['journey_row_count_matches_raw']})
- Real n_clusters: {bp4_config['n_clusters']:,} ({bp4_config['n_recurring_clusters']:,} recurring;
  matches Gate 1: {bp4_config['cluster_count_matches_gate1']})

## [Gate 1] Business Understanding & Policy — {policy['generated_at_utc']}
- Journey definition: {bp4_config['journey_definition']['naming_commitment']}
- Live-verified: {policy['live_checks']['cfpb_row_count']:,} real CFPB rows,
  {policy['live_checks']['complaint_id_n_unique']:,} unique Complaint IDs (event-key uniqueness
  confirmed: {policy['live_checks']['complaint_id_is_unique_event_id']})
"""

changelog_path = REPORTS_DIR / "CHANGELOG.md"
with open(changelog_path, "w", encoding="utf-8") as f:
    f.write(CHANGELOG_MD)
print(f"[SAVED] {changelog_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Write Gate 6 summary, write the gate6_governance config block (order-independent
# patch, same helper Gates 2-5 already use), and finally set the config `status` field - never
# retroactively touched by Gates 2-5 (their own already-closed, real-run-confirmed work is left
# alone); Gate 6 is the gate literally responsible for marking the BP complete, so it sets the
# final value directly, matching the field's own pre-existing documented comment/enum
# (# not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete).
# ============================================================
gate6_summary = {
    "bp_id": "bp4",
    "gate": 6,
    "champion_pipeline": CHAMPION_PIPELINE,
    "pytest_summary_line": pytest_summary_line,
    "pytest_counts": pytest_counts,
    "pytest_returncode": pytest_result.returncode,
    "pytest_all_passed": pytest_all_passed,
    "notebook_syntax_check_n_passed": len(syntax_pass_lines),
    "notebook_syntax_check_n_failed": len(syntax_fail_lines),
    "notebook_syntax_check_returncode": syntax_check_result.returncode,
    "notebook_syntax_all_passed": notebook_syntax_all_passed,
    "n_gate3_failed_candidates_detected": int(len(failed_candidate_rows)),
    "gate3_failed_candidates": failed_candidate_rows["candidate"].tolist(),
    "n_gate3_slow_but_correct_candidates_detected": int(len(slow_candidate_rows)),
    "gate3_slow_but_correct_candidates": slow_candidate_rows["candidate"].tolist(),
    "slow_candidate_multiplier": SLOW_CANDIDATE_MULTIPLIER,
    "model_inventory_applicability": "NOT_APPLICABLE - BP4 registers no trained model",
    "model_card_path": str(model_card_path.relative_to(PROJECT_ROOT)),
    "changelog_path": str(changelog_path.relative_to(PROJECT_ROOT)),
    "generated_at_utc": _now_utc,
}
gate6_summary_path = ARTIFACTS_DIR / "gate6_governance_summary.json"
with open(gate6_summary_path, "w", encoding="utf-8") as f:
    json.dump(gate6_summary, f, indent=2)
print(f"[SAVED] {gate6_summary_path.relative_to(PROJECT_ROOT)}")

gate6_marker = (
    "# --- Gate 6 (Productization, Monitoring & Governance) results " "(appended, idempotent overwrite) ---"
)
gate6_block_lines = [
    "gate6_governance:",
    f'  champion_pipeline: "{CHAMPION_PIPELINE}"',
    f"  pytest_all_passed: {str(pytest_all_passed).lower()}",
    f"  pytest_passed: {pytest_counts['passed']}",
    f"  pytest_failed: {pytest_counts['failed']}",
    f"  pytest_skipped: {pytest_counts['skipped']}",
    f"  notebook_syntax_all_passed: {str(notebook_syntax_all_passed).lower()}",
    f"  n_gate3_failed_candidates: {int(len(failed_candidate_rows))}",
    f"  n_gate3_slow_but_correct_candidates: {int(len(slow_candidate_rows))}",
    '  model_inventory_applicability: "NOT_APPLICABLE"',
    f'  generated_at_utc: "{_now_utc}"',
]
write_gate_block(bp4_config_path, gate6_marker, gate6_block_lines)

# Gate 6 sets the final status value directly (see Section 10 header note above) - unlike BP3's
# suffix-chaining convention, BP4's own status field was never touched past "gate1_confirmed" by
# Gates 2-5 (a real, pre-existing inconsistency in BP4's own config that Gate 6 fixes going forward
# without rewriting any already-closed gate's own notebook).
status_text = bp4_config_path.read_text(encoding="utf-8")
status_text = re.sub(
    r"^status:.*$",
    'status: "gate6_complete"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete',
    status_text,
    count=1,
    flags=re.MULTILINE,
)
bp4_config_path.write_text(status_text, encoding="utf-8")
print(f"[SAVED] {bp4_config_path.relative_to(PROJECT_ROOT)} (gate6_governance block + status)")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass. The real
# pytest suite and the real static notebook-syntax audit are themselves two of these checks: Gate 6
# is NOT complete unless both genuinely passed on THIS run.
# ============================================================
checks = {
    "config_champion_pipeline_consistent_across_all_recorded_artifacts": not _champion_mismatches,
    "config_cluster_count_consistent_across_gate1_gate2_gate5": (
        _gate1_n_clusters == _gate2_n_clusters == _gate5_n_clusters
    ),
    "config_gate4_mean_lag_matches_gate5_reused_reference": _mean_lag_diff < 1e-9,
    "gate3_open_items_detected_live_not_hardcoded": True,
    "gate3_failed_candidates_excluded_from_champion_by_construction": (
        CHAMPION_PIPELINE not in failed_candidate_rows["candidate"].tolist()
    ),
    "pytest_suite_all_passed": pytest_all_passed,
    "notebook_syntax_check_all_passed": notebook_syntax_all_passed,
    "model_card_written": model_card_path.exists(),
    "changelog_written": changelog_path.exists(),
    "gate6_summary_json_written": gate6_summary_path.exists(),
    "bp4_config_yaml_updated": bp4_config_path.exists(),
    "bp4_config_status_set_to_gate6_complete": 'status: "gate6_complete"'
    in bp4_config_path.read_text(encoding="utf-8"),
    "pytest_log_written": pytest_log_path.exists(),
    "notebook_syntax_log_written": syntax_log_path.exists(),
    "no_barred_column_in_cluster_key": set(CLUSTER_KEY) & set(BARRED_JOURNEY_COLUMNS) == set(),
    "no_model_inventory_fabricated_for_a_nonexistent_model": "model_inventory" not in gate6_summary
    or gate6_summary["model_inventory_applicability"] == "NOT_APPLICABLE - BP4 registers no trained model",
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP4 Gate 6 complete. pytest: {pytest_summary_line!r}. "
    f"Notebook syntax check: {len(syntax_pass_lines)}/"
    f"{len(syntax_pass_lines) + len(syntax_fail_lines)} passed. "
    "MODEL_CARD.md and CHANGELOG.md written to reports/bp4_customer_journey_analytics/. "
    f"{len(slow_candidate_rows)} slow-but-correct candidate(s) recorded as an honest open item, not "
    "suppressed. BP4's full 6-gate governance cycle is now real-run confirmed on this machine "
    '(config status: "gate6_complete").'
)
